# Prepare Datasets for Machine Learning

In previous notebooks, SNOTEL, PRISM, and DEM datasets were downloaded and cleaned in preparation for application to machine learning. This notebook will perform final cleaning and calculate some further statistics for the PRISM dataset; the following notebook will perform the machine learning.

## Step 1: Import Libraries and Set Up Project Directory

In [90]:
# import libraries

# file management
import os
import pathlib
from pathlib import Path
import sys

# datatypes
import numpy as np
import pandas as pd
import xarray as xr

# geospatial data
import geopandas as gpd
import rioxarray as rxr

# plotting
import matplotlib
import matplotlib.pyplot as plt
import holoviews as hv
import hvplot.pandas

In [2]:
# set directories
proj_dir = os.path.join(pathlib.Path.home(),
                        'Documents',
                        'Graduate_School',
                        'EDA_Certificate', 
                        'Summer', 
                        'snow-drought-modeling')
os.makedirs(proj_dir, exist_ok=True)

raw_data_dir = os.path.join(proj_dir, 'data', 'raw')
os.makedirs(raw_data_dir, exist_ok=True)

cleaned_data_dir = os.path.join(proj_dir, 'data', 'cleaned')
os.makedirs(raw_data_dir, exist_ok=True)

## Step 2: Station Filtering

This step will filter the SNOTEL stations used in the ML model to stations with at least 20 years of record keeping. Ideally, stations with 30 years would be used. However, this will likely disqualify too many stations and reduce the dataset size too much.

#### Step 2a: Import PRISM

In [7]:
# Import PRISM dataset

# set a path
prism_path = Path(cleaned_data_dir, 'prism', 'prism_mhw_1990_2020_cleaned_with_crs.nc')

# open prism dataset
prism_ds = xr.open_dataset(prism_path, decode_coords='all')

# check on DS
print(prism_ds.rio.crs)
prism_ds

EPSG:4326


<xarray.Dataset> Size: 372MB
Dimensions:  (time: 7328, lat: 51, lon: 83)
Coordinates:
  * lon      (lon) float64 664B -113.9 -113.9 -113.8 ... -110.6 -110.5 -110.5
  * lat      (lat) float64 408B 46.46 46.42 46.38 46.33 ... 44.46 44.42 44.38
  * time     (time) datetime64[ns] 59kB 1990-10-01 1990-10-02 ... 2020-06-01
    crs      int64 8B ...
Data variables:
    ppt      (time, lat, lon) float32 124MB ...
    tmin     (time, lat, lon) float32 124MB ...
    tmax     (time, lat, lon) float32 124MB ...
Attributes:
    Conventions:  CF-1.5
    GDAL:         GDAL 3.12.0 "Chicoutimi", released 2025/11/03
    history:      Thu Jun 18 17:05:52 2026: GDAL CreateCopy( /nfs/pancake/u5/...

#### Step 2b: Import SNOTEL

In [6]:
# Import SNOTEL dataset

# set a path
snotel_path = Path(cleaned_data_dir, 'bcqc_snotel_1990-2020_RAW.nc')

# open SNOTEL dataset
snotel_ds = xr.open_dataset(snotel_path, decode_coords='all')

# check ds
snotel_ds

<xarray.Dataset> Size: 11MB
Dimensions:          (stationTriplet: 26, date: 10837)
Coordinates:
  * date             (date) datetime64[ns] 87kB 1990-10-01 ... 2020-06-01
  * stationTriplet   (stationTriplet) <U11 1kB '916:MT:SNTL' ... '384:WY:SNTL'
    stationId        (stationTriplet) <U3 312B ...
    name             (stationTriplet) <U16 2kB ...
    latitude         (stationTriplet) float64 208B ...
    longitude        (stationTriplet) float64 208B ...
    beginDate        (stationTriplet) <U16 2kB ...
    endDate          (stationTriplet) <U16 2kB ...
Data variables:
    daily_precip_in  (stationTriplet, date) float64 2MB ...
    tmax_f           (stationTriplet, date) float64 2MB ...
    tmin_f           (stationTriplet, date) float64 2MB ...
    tavg_f           (stationTriplet, date) float64 2MB ...
    SWE              (stationTriplet, date) float64 2MB ...

In [29]:
# check units

print(snotel_ds.isel(stationTriplet=slice(0, 2), date=slice(70, 73)).to_dataframe())

                           daily_precip_in  tmax_f  tmin_f  tavg_f  SWE  \
stationTriplet date                                                       
916:MT:SNTL    1990-12-10              NaN     NaN     NaN     NaN  NaN   
               1990-12-11              NaN     NaN     NaN     NaN  NaN   
               1990-12-12              NaN     NaN     NaN     NaN  NaN   
318:MT:SNTL    1990-12-10              0.0   38.62   28.32   32.44  1.6   
               1990-12-11              0.1   30.38   -0.52   19.05  1.6   
               1990-12-12              0.1   13.90   -4.64    1.54  1.6   

                          stationId            name  latitude  longitude  \
stationTriplet date                                                        
916:MT:SNTL    1990-12-10       916      Albro Lake  45.59723 -111.95902   
               1990-12-11       916      Albro Lake  45.59723 -111.95902   
               1990-12-12       916      Albro Lake  45.59723 -111.95902   
318:MT:SNTL    1990

Units are all imperial. The next code will add data variables that have been converted to metric.

In [ ]:
# Unit Conversion

# set conversion factor
inch_to_mm = 25.4

# Perform conversions
snotel_ds = snotel_ds.assign(
    # Temp conversion
    tavg_c = (snotel_ds['tavg_f'] - 32) * (5/9),
    tmin_c = (snotel_ds['tmin_f'] - 32) * (5/9),
    tmax_c = (snotel_ds['tmax_f'] - 32) * (5/9),

    # Precip conversion
    daily_precip_mm = snotel_ds['daily_precip_in'] * inch_to_mm,
    swe_mm = snotel_ds['SWE'] * inch_to_mm
)

#### Step 2c: Filter PRISM at SNOTEL sites

In [ ]:
# extract prism values at station locations

# initialize list
prism_filtered_list = []

# get length of stations
num_stations = len(snotel_ds.stationTriplet)

# filter through prism and extract timeseries for each pixel that has a snotel station in it
for i in range(num_stations):
    st_id = snotel_ds['stationTriplet'].values[i]
    lat = float(snotel_ds['latitude'].values[i])
    lon = float(snotel_ds['longitude'].values[i])

    # filter prism dataset
    prism_pixel = prism_ds.sel(
        # grab pixel that matches station coords
        lon = lon, lat = lat,
        # get the pixel nearest to the station
        method='nearest'
        # convert to DF
        ).to_dataframe().reset_index()
    
    # keep only climate and time data
    prism_pixel_df = prism_pixel[
        # grab just time, ppt, tmin, tmax
        ['time', 'ppt', 'tmin', 'tmax']
        # rename columns to remind that it's prism data
        ].rename(
            columns = {
                'time': 'date',
                'ppt': 'prism_ppt_mm',
                'tmin': 'prism_tmin_c',
                'tmax': 'prism_tmax_c'
            }
        )
    
    # append station Triplet to it can be joined to the SNOTEL dataset
    prism_pixel_df['stationTriplet'] = st_id

    # append to list
    prism_filtered_list.append(prism_pixel_df)

# concat prism dfs
prism_filter_df = pd.concat(prism_filtered_list, ignore_index=True)

In [40]:
# check the head and tail
print(prism_filter_df.head())
print(prism_filter_df.tail())

        date  prism_ppt_mm  prism_tmin_c  prism_tmax_c stationTriplet
0 1990-10-01         0.000         0.324        14.996    916:MT:SNTL
1 1990-10-02         0.000         2.630        15.985    916:MT:SNTL
2 1990-10-03         1.948        -6.244         4.975    916:MT:SNTL
3 1990-10-04         0.000        -6.558         2.983    916:MT:SNTL
4 1990-10-05         1.476         2.839        12.372    916:MT:SNTL
             date  prism_ppt_mm  prism_tmin_c  prism_tmax_c stationTriplet
190523 2020-05-28         0.000         0.379     17.624001    384:WY:SNTL
190524 2020-05-29         0.000         0.574     20.450001    384:WY:SNTL
190525 2020-05-30         0.000         1.982     22.357000    384:WY:SNTL
190526 2020-05-31         0.072         3.837     24.507000    384:WY:SNTL
190527 2020-06-01         0.000         2.544     19.605000    384:WY:SNTL


#### Step 2d: Count how many years of data each SNOTEL station has

In [ ]:
# check start years

# get start year values
start_years = snotel_ds.beginDate.values

# convert to datetime
start_years_dt = pd.to_datetime(start_years)

# check how many stations started by 1990 and by 1996
# that's 30 or 25 years of data, respectively
print(sum(start_years_dt <= '1990-10-01'))
print(sum(start_years_dt <= '1996-10-01'))

# check overall number of stations
print(len(start_years))

23
25
26


Starting by 1990 keeps 23 stations, while starting five years later keeps 25 stations. One station is lost either way. Preserving spatial resolution seems more important than preserving temporal resolution, so I'll filter to stations starting by 1996.

In [58]:
# set mask based on station start date
station_mask = start_years_dt <= pd.to_datetime('1996-10-01')

# filter ds
snotel_filter_ds = snotel_ds.sel(stationTriplet = station_mask)

# check that filtering worked
print(f"snotel_ds has {len(snotel_ds.stationTriplet)} stations")
print(f"snotel_filter_ds has {len(snotel_filter_ds.stationTriplet)} stations")

snotel_ds has 26 stations
snotel_filter_ds has 25 stations


#### Step 2e: Convert SNOTEL to DataFrame

In [108]:
# convert from DS to DF
snotel_full_df = snotel_filter_ds.to_dataframe().reset_index()

# remove 1990-1995 years (the previous step didn't filter temporally)
# set date to datetime
snotel_full_df['date'] = pd.to_datetime(snotel_full_df['date'])

# set start and end dates
start_date = '1996-10-01'
end_date = '2020-06-01'

# filter using start and end dates
snotel_time_filter_df = snotel_full_df[(snotel_full_df['date'] >= start_date) & (snotel_full_df['date'] <= end_date)]

# filter out imperial columns
# set columns to keep
columns_keep = [
    # metadata
    'stationTriplet', 'date', 'latitude', 'longitude',
    # data
    'tavg_c', 'tmin_c', 'tmax_c', 'daily_precip_mm', 'swe_mm']

# filter
snotel_df = snotel_time_filter_df[columns_keep]

# check it out
print(snotel_df.head())
print(snotel_df.tail())

     stationTriplet       date  latitude  longitude    tavg_c    tmin_c  \
2192    916:MT:SNTL 1996-10-01  45.59723 -111.95902  5.394444 -3.761111   
2193    916:MT:SNTL 1996-10-02  45.59723 -111.95902  7.111111 -1.472222   
2194    916:MT:SNTL 1996-10-03  45.59723 -111.95902  8.255556  3.105556   
2195    916:MT:SNTL 1996-10-04  45.59723 -111.95902  9.972222  3.105556   
2196    916:MT:SNTL 1996-10-05  45.59723 -111.95902  8.827778  1.388889   

         tmax_c  daily_precip_mm  swe_mm  
2192  12.833333              0.0     0.0  
2193  16.838889              0.0     0.0  
2194  16.838889              0.0     0.0  
2195  16.266667              0.0     0.0  
2196  13.977778              0.0     0.0  
       stationTriplet       date  latitude  longitude     tavg_c    tmin_c  \
270920    384:WY:SNTL 2020-05-28  44.71961 -110.51084  10.544444 -0.327778   
270921    384:WY:SNTL 2020-05-29  44.71961 -110.51084  12.261111  1.388889   
270922    384:WY:SNTL 2020-05-30  44.71961 -110.51084  13

## Step 3: Geospatial Joins

This step will join the PRISM, SNOTEL, and DEM datasets into a single dataframe.

#### Step 3a: Join PRISM and SNOTEL datasets

In [109]:
# merge PRISM and SNOTEL datasets
prism_snotel_df = pd.merge(
    snotel_df,
    prism_filter_df,
    on = ['stationTriplet', 'date'],
    how = 'inner'
)

# check it out
prism_snotel_df

,stationTriplet,date,latitude,longitude,tavg_c,tmin_c,tmax_c,daily_precip_mm,swe_mm,prism_ppt_mm,prism_tmin_c,prism_tmax_c
0,916:MT:SNTL,1996-10-01,45.59723,-111.95902,5.394444,-3.761111,12.833333,0.0,0.00,0.000,4.278,16.568001
1,916:MT:SNTL,1996-10-02,45.59723,-111.95902,7.111111,-1.472222,16.838889,0.0,0.00,0.001,-2.904,13.110000
2,916:MT:SNTL,1996-10-03,45.59723,-111.95902,8.255556,3.105556,16.838889,0.0,0.00,0.000,-1.004,16.767000
3,916:MT:SNTL,1996-10-04,45.59723,-111.95902,9.972222,3.105556,16.266667,0.0,0.00,0.000,2.815,16.663000
4,916:MT:SNTL,1996-10-05,45.59723,-111.95902,8.827778,1.388889,13.977778,0.0,0.00,0.000,3.384,16.459999
...,...,...,...,...,...,...,...,...,...,...,...,...
146545,384:WY:SNTL,2020-05-28,44.71961,-110.51084,10.544444,-0.327778,19.700000,0.0,10.16,0.000,0.379,17.624001
146546,384:WY:SNTL,2020-05-29,44.71961,-110.51084,12.261111,1.388889,21.416667,0.0,0.00,0.000,0.574,20.450001
146547,384:WY:SNTL,2020-05-30,44.71961,-110.51084,13.977778,2.533333,23.705556,0.0,0.00,0.000,1.982,22.357000
146548,384:WY:SNTL,2020-05-31,44.71961,-110.51084,12.833333,3.677778,19.127778,0.0,0.00,0.072,3.837,24.507000


#### Step 3b: Import DEM datasets

In [86]:
# import DEM datasets

# set topo dir
topo_rsmp_dir = Path(cleaned_data_dir, 'topo')

# paths
elev_4km_path = Path(topo_rsmp_dir, 'mhw_elevation_4km.tif')
slope_4km_path = Path(topo_rsmp_dir, 'mhw_slope_4km.tif')
aspect_4km_path = Path(topo_rsmp_dir, 'mhw_aspect_4km.tif')
aspect_north_4km_path = Path(topo_rsmp_dir, 'mhw_northness_4km.tif')
aspect_east_4km_path = Path(topo_rsmp_dir, 'mhw_eastness_4km.tif')

# import
mhw_elev_4km_da = rxr.open_rasterio(elev_4km_path)
mhw_slope_4km_da = rxr.open_rasterio(slope_4km_path)
mhw_aspect_4km_da = rxr.open_rasterio(aspect_4km_path)
mhw_aspect_4km_north_da = rxr.open_rasterio(aspect_north_4km_path)
mhw_aspect_4km_east_da = rxr.open_rasterio(aspect_east_4km_path)

# merge into one dataset
dem_dataset = xr.Dataset({
    'elevation': mhw_elev_4km_da,
    'slope': mhw_slope_4km_da,
    'aspect': mhw_aspect_4km_da,
    'aspect_north': mhw_aspect_4km_north_da,
    'aspect_east': mhw_aspect_4km_east_da
})

# make sure dataset is in ESPG: 4326
if dem_dataset.rio.crs.to_epsg() != 4326:
    dem_dataset = dem_dataset.rio.reproject("EPSG:4326")

#### Step 3c: Select DEM data at stations

In [99]:
# grab each station's point
stations_point_gdf = (prism_snotel_gdf[['stationTriplet', 'latitude', 'longitude', 'geometry']]
                     # get rid of duplicate lines, only need one row per station
                     .drop_duplicates().reset_index())

# grab coordinates for each station
x_coords = xr.DataArray(
    stations_point_gdf['longitude'], 
    dims="stationTriplet", 
    coords={"stationTriplet": stations_point_gdf['stationTriplet']})
y_coords = xr.DataArray(
    stations_point_gdf['latitude'], 
    dims="stationTriplet", 
    coords={"stationTriplet": stations_point_gdf['stationTriplet']})

# search through DEM dataset and grab all stats for each point
station_dem_ds = dem_dataset.sel(x = x_coords, y = y_coords, method = 'nearest')
station_dem_ds = station_dem_ds.squeeze('band', drop=True)

# convert to dataframe before merging
station_dem_df = station_dem_ds.to_dataframe().reset_index()

# drop extra columns
station_dem_df = station_dem_df[['stationTriplet', 'elevation', 'slope', 'aspect_north', 'aspect_east']]

# check it out
station_dem_df

,stationTriplet,elevation,slope,aspect_north,aspect_east
0,916:MT:SNTL,2546,20.723917,0.208108,0.978106
1,318:MT:SNTL,2996,21.693270,-0.455876,0.890043
2,328:MT:SNTL,2532,14.942879,-0.008403,-0.999965
3,347:MT:SNTL,2479,2.508581,0.843661,0.536875
4,355:MT:SNTL,2343,8.982809,-0.242535,-0.970143
5,365:MT:SNTL,2372,23.627708,-0.032591,-0.999469
6,381:MT:SNTL,1970,4.599943,-0.728200,0.685365
7,385:MT:SNTL,2722,9.332155,-0.977648,0.210247
8,403:MT:SNTL,2540,23.873613,0.955363,-0.295434
9,436:MT:SNTL,2718,18.510357,0.055470,0.998460


#### 3d: Join DEM with PRISM/SNOTEL df

In [111]:
# join on station triplet
stations_ml_df = pd.merge(
    prism_snotel_df,
    station_dem_df,
    on = 'stationTriplet',
    how = 'left'
)

# check it out
stations_ml_df

,stationTriplet,date,latitude,longitude,tavg_c,tmin_c,tmax_c,daily_precip_mm,swe_mm,prism_ppt_mm,prism_tmin_c,prism_tmax_c,elevation,slope,aspect_north,aspect_east
0,916:MT:SNTL,1996-10-01,45.59723,-111.95902,5.394444,-3.761111,12.833333,0.0,0.00,0.000,4.278,16.568001,2546,20.723917,0.208108,0.978106
1,916:MT:SNTL,1996-10-02,45.59723,-111.95902,7.111111,-1.472222,16.838889,0.0,0.00,0.001,-2.904,13.110000,2546,20.723917,0.208108,0.978106
2,916:MT:SNTL,1996-10-03,45.59723,-111.95902,8.255556,3.105556,16.838889,0.0,0.00,0.000,-1.004,16.767000,2546,20.723917,0.208108,0.978106
3,916:MT:SNTL,1996-10-04,45.59723,-111.95902,9.972222,3.105556,16.266667,0.0,0.00,0.000,2.815,16.663000,2546,20.723917,0.208108,0.978106
4,916:MT:SNTL,1996-10-05,45.59723,-111.95902,8.827778,1.388889,13.977778,0.0,0.00,0.000,3.384,16.459999,2546,20.723917,0.208108,0.978106
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146545,384:WY:SNTL,2020-05-28,44.71961,-110.51084,10.544444,-0.327778,19.700000,0.0,10.16,0.000,0.379,17.624001,2387,7.114066,-0.404747,-0.914429
146546,384:WY:SNTL,2020-05-29,44.71961,-110.51084,12.261111,1.388889,21.416667,0.0,0.00,0.000,0.574,20.450001,2387,7.114066,-0.404747,-0.914429
146547,384:WY:SNTL,2020-05-30,44.71961,-110.51084,13.977778,2.533333,23.705556,0.0,0.00,0.000,1.982,22.357000,2387,7.114066,-0.404747,-0.914429
146548,384:WY:SNTL,2020-05-31,44.71961,-110.51084,12.833333,3.677778,19.127778,0.0,0.00,0.072,3.837,24.507000,2387,7.114066,-0.404747,-0.914429


## Step 4: NaN Removal

This step will remove any remaining NaNs from the PRISM and SNOTEL datasets using interpolation or regression. 

I can't just use .dropna(), as I need to preserve the daily structure of the record. Thus, some type of interpolation is required. However, the methodology is important here, as temp and precip can't be interpolated in the same ways, given their inherent differences.



In [112]:
# check for NaNs
print(stations_ml_df.isna().sum())

stationTriplet         0
date                   0
latitude               0
longitude              0
tavg_c              1250
tmin_c              1084
tmax_c              1045
daily_precip_mm    16132
swe_mm             15643
prism_ppt_mm           0
prism_tmin_c           0
prism_tmax_c           0
elevation              0
slope                  0
aspect_north           0
aspect_east            0
dtype: int64


In [113]:
# sort df by station triplet and date 
stations_ml_df =stations_ml_df.sort_values(by=['stationTriplet', 'date']).reset_index(drop=True)

In [120]:
# temporarily set index as date
stations_ml_df = stations_ml_df.set_index('date')

# time-based interpolation for temperature variables
temp_vars = ['tavg_c', 'tmin_c', 'tmax_c', 'prism_tmin_c', 'prism_tmax_c']
stations_ml_df[temp_vars] = stations_ml_df.groupby('stationTriplet')[temp_vars].transform(
    lambda x: x.interpolate(method='time'))

In [117]:
# linear interpolation for SWE
stations_ml_df['swe_mm'] = stations_ml_df.groupby('stationTriplet')['swe_mm'].transform(
    lambda x: x.interpolate(method='linear'))

In [121]:
# Patch SNOTEL precip values with PRISM data

# reset index
stations_ml_df = stations_ml_df.reset_index()

# create a flag for days where SNOTEL precip data was NaN
stations_ml_df['precip_patched_flag'] = stations_ml_df['daily_precip_mm'].isna().astype(int)

# patch missing SNOTEL values with PRISM data
stations_ml_df['daily_precip_mm'] = stations_ml_df['daily_precip_mm'].fillna(stations_ml_df['prism_ppt_mm'])

In [122]:
# check again for NaNs
print(stations_ml_df.isna().sum())

date                      0
stationTriplet            0
latitude                  0
longitude                 0
tavg_c                  488
tmin_c                  488
tmax_c                  488
daily_precip_mm           0
swe_mm                 2929
prism_ppt_mm              0
prism_tmin_c              0
prism_tmax_c              0
elevation                 0
slope                     0
aspect_north              0
aspect_east               0
precip_patched_flag       0
dtype: int64


Using interpolation can't patch all of the missing data, unfortunately. These last NaNs will be dropped at the very end of the prep process.

## Step 5: Statistics Calculations

This step will calculate additional statistics on the PRISM and SNOTEL stations, such as rolling averages of temperature and precipitation. [Moya et al. 2026](https://doi.org/10.5194/tc-20-1427-2026) found that lagged variables were impactful, but not by much after 3 days for air temp and certainly not helpful after 7 days. 

In [ ]:
# set station df index to date so Pandas can calculate rolling windows
stations_ml_df = stations_ml_df.set_index('date')

# rolling 3 day window of  mean PRISM tmin
stations_ml_df['prism_tmin_c_roll_3d'] = (stations_ml_df
                                          # group by each station and tmin
                                          .groupby('stationTriplet')['prism_tmin_c']
                                          # calculate rolling three day mean
                                          .transform(
                                              lambda x: x.rolling('3D').mean
                                          )
                                          )

# rolling 3 day window of mean PRISM tmax
stations_ml_df['prism_tmax_c_roll_3d'] = (stations_ml_df
                                          # group by each station and tmin
                                          .groupby('stationTriplet')['prism_tmax_c']
                                          # calculate rolling three day mean
                                          .transform(
                                              lambda x: x.rolling('3D').mean
                                          )
                                          )

## Step 6: Train/Test Splitting

This step will perform train/test splitting for use in the machine learning model. Because the data is so temporal in nature, the train/test split will be performed by removing the last 5 years of the dataset (20% of the data). 

## Step 7: Save dataset for use in the model